CUDA Testing to build model on Graphic Card

In [1]:
import tensorflow as tf
import numpy as np
import torch

print(f"Wersja TensorFlow: {tf.__version__}")
print(f"Wersja NumPy: {np.__version__}")
print("---")

gpus = tf.config.list_physical_devices('GPU')
if gpus:
    print(f"SUKCES! Znaleziono GPU: {gpus}")
else:
    print("BŁĄD: Nadal nie widzę GPU. Sprawdź czy foldery bin z CUDA i cuDNN są w zmiennych środowiskowych PATH.")


print(torch.cuda.is_available())

TypeError: Descriptors cannot be created directly.
If this call came from a _pb2.py file, your generated code is out of date and must be regenerated with protoc >= 3.19.0.
If you cannot immediately regenerate your protos, some other possible workarounds are:
 1. Downgrade the protobuf package to 3.20.x or lower.
 2. Set PROTOCOL_BUFFERS_PYTHON_IMPLEMENTATION=python (but this will use pure-Python parsing and will be much slower).

More information: https://developers.google.com/protocol-buffers/docs/news/2022-05-06#python-updates

Confussion Matrix for YOLOv8

In [2]:
import cv2
import os
from ultralytics import YOLO

if __name__ == '__main__':
    model_path = r"D:\Studia\cybAIR\Ambition\rocks_detection\Detection\runs\detect\depthai_model\yolo_rocks_btr_noise\weights\best.pt"
    data_yaml_path = r"D:\Studia\cybAIR\Ambition\rocks_detection\Detection\data.yaml"

    model = YOLO(model_path)

    metrics = model.val(data=data_yaml_path)

    print(f"Wyniki zapisano w katalogu: {metrics.save_dir}")

    matrix_path = os.path.join(metrics.save_dir, "confusion_matrix.png")

    if os.path.exists(matrix_path):
        img = cv2.imread(matrix_path)

        img_resized = cv2.resize(img, (1024, 768))

        cv2.imshow("Macierz Bledow", img_resized)
        cv2.waitKey(0)
        cv2.destroyAllWindows()
    else:
        print("Nie znaleziono pliku z macierza bledow.")

Ultralytics 8.4.52  Python-3.10.11 torch-2.7.1+cu118 CUDA:0 (GeForce GTX 1650 Ti, 4096MiB)
Model summary (fused): 73 layers, 3,005,843 parameters, 0 gradients, 8.1 GFLOPs
val: Fast image access  (ping: 0.10.0 ms, read: 1800.6810.7 MB/s, size: 5781.6 KB)
val: Scanning D:\Studia\cybAIR\Ambition\rocks_detection\Detection\valid\labels.cache... 115 images, 9 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 115/115  0.0s
val: D:\Studia\cybAIR\Ambition\rocks_detection\Detection\valid\images\20260601_174934_jpg.rf.LzfJioODWeDyZ4C3ZEiF.jpg: corrupt JPEG restored and saved
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 8/8 1.8it/s 4.5s0.4s
                   all        115       2227       0.82      0.689      0.766      0.502
Speed: 3.1ms preprocess, 11.1ms inference, 0.0ms loss, 13.3ms postprocess per image
Results saved to D:\Studia\cybAIR\Ambition\rocks_detection\Detection\runs\detect\val-12
Wyniki zapisano w katalogu: D:\Studia\cybAIR\Am

Convert YOLO to .blob

In [ ]:
import os
from ultralytics import YOLO
import blobconverter

model = YOLO(r"D:\Studia\cybAIR\Ambition\rocks_detection\Detection\runs\detect\depthai_model\yolo_rocks-5\weights\best.pt")

onnx_model_path = model.export(format="onnx", imgsz=416, opset=11, nms=False)
print(f"Model wyeksportowany do ONNX: {onnx_model_path}")

blob_path = blobconverter.from_onnx(
    model=onnx_model_path,
    data_type="FP16",
    shaves=6,
    version="2022.1",
    output_dir="depthai_model"
)

print(f"Sukces! Plik skompilowany do OAK-D Lite: {blob_path}")

Confussion Matrix for .blob

In [ ]:
#nothing here

Convert PNG Labels to TXT

In [2]:
import os
import cv2

def convert_masks_to_yolo_seg(masks_dir, labels_output_dir, class_id=0):
    os.makedirs(labels_output_dir, exist_ok=True)

    for mask_name in os.listdir(masks_dir):
        if mask_name.lower().endswith('.png'):
            mask_path = os.path.join(masks_dir, mask_name)
            mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)

            if mask is None:
                continue

            h, w = mask.shape
            _, thresh = cv2.threshold(mask, 127, 255, cv2.THRESH_BINARY)
            contours, _ = cv2.findContours(thresh, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

            txt_lines = []
            for contour in contours:
                if cv2.contourArea(contour) < 10:
                    continue

                polygon_coords = []
                for point in contour:
                    pt_x, pt_y = point[0]
                    norm_x = pt_x / w
                    norm_y = pt_y / h
                    polygon_coords.append(f"{norm_x:.6f} {norm_y:.6f}")

                if polygon_coords:
                    coords_str = " ".join(polygon_coords)
                    txt_lines.append(f"{class_id} {coords_str}")

            if txt_lines:
                txt_name = os.path.splitext(mask_name)[0] + ".txt"
                txt_path = os.path.join(labels_output_dir, txt_name)
                with open(txt_path, "w") as f:
                    f.write("\n".join(txt_lines))

if __name__ == "__main__":
    base_dir = r"D:\Studia\cybAIR\Ambition\rocks_detection\Detection\notTested"

    splits = ["train", "val", "test"]

    for split in splits:
        masks_folder = os.path.join(base_dir, split, "images")
        labels_folder = os.path.join(base_dir, split, "labels")

        if os.path.exists(masks_folder):
            print(f"Przetwarzanie masek dla zestawu: {split}...")
            convert_masks_to_yolo_seg(masks_folder, labels_folder, class_id=0)

    print("Wszystkie maski zostały przekonwertowane do formatu YOLO-Segmentation!")

Przetwarzanie masek dla zestawu: train...
Przetwarzanie masek dla zestawu: val...
Przetwarzanie masek dla zestawu: test...
Wszystkie maski zostały przekonwertowane do formatu YOLO-Segmentation!


Converting our area detection of object to rectangle

In [1]:
import os

def convert_polygon_to_bbox(labels_dir):
    if not os.path.exists(labels_dir):
        return

    for filename in os.listdir(labels_dir):
        if not filename.endswith(".txt"):
            continue

        filepath = os.path.join(labels_dir, filename)

        with open(filepath, "r") as f:
            lines = f.readlines()

        new_lines = []
        for line in lines:
            parts = line.strip().split()
            if len(parts) < 5:
                new_lines.append(line.strip())
                continue

            class_id = parts[0]

            coords = [float(x) for x in parts[1:]]
            xs = coords[0::2]
            ys = coords[1::2]

            if not xs or not ys:
                continue

            x_min, x_max = min(xs), max(xs)
            y_min, y_max = min(ys), max(ys)

            x_center = (x_min + x_max) / 2.0
            y_center = (y_min + y_max) / 2.0
            width = x_max - x_min
            height = y_max - y_min

            new_lines.append(f"{class_id} {x_center:.6f} {y_center:.6f} {width:.6f} {height:.6f}")

        with open(filepath, "w") as f:
            f.write("\n".join(new_lines))

if __name__ == "__main__":
    base_dir = "."

    splits = ["train", "valid", "test"]

    for split in splits:
        labels_folder = os.path.join(base_dir, split, "labels")
        if os.path.exists(labels_folder):
            convert_polygon_to_bbox(labels_folder)

Increase data with noise

In [1]:
import os
import cv2
import shutil
import albumentations as A

IMG_DIR = './train/images'
LBL_DIR = './train/labels'
OUT_IMG_DIR = './trainNoise/images'
OUT_LBL_DIR = './trainNoise/labels'

os.makedirs(OUT_IMG_DIR, exist_ok=True)
os.makedirs(OUT_LBL_DIR, exist_ok=True)

transform = A.Compose([
    A.GaussNoise(std_range=(0.1, 0.25), p=0.7),
    A.MotionBlur(blur_limit=7, p=0.4),
    A.RandomBrightnessContrast(brightness_limit=0.3, contrast_limit=0.3, p=0.7),
    A.PixelDropout(dropout_prob=0.01, p=0.3)
])

for img_name in os.listdir(IMG_DIR):
    if not img_name.lower().endswith(('.png', '.jpg', '.jpeg')):
        continue

    img_path = os.path.join(IMG_DIR, img_name)
    image = cv2.imread(img_path)

    if image is None:
        continue

    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

    augmented = transform(image=image)
    aug_image = augmented['image']

    aug_image = cv2.cvtColor(aug_image, cv2.COLOR_RGB2BGR)

    base_name = os.path.splitext(img_name)[0]
    new_img_name = f"{base_name}_aug.jpg"
    new_lbl_name = f"{base_name}_aug.txt"

    new_img_path = os.path.join(OUT_IMG_DIR, new_img_name)
    cv2.imwrite(new_img_path, aug_image)

    lbl_path = os.path.join(LBL_DIR, f"{base_name}.txt")
    new_lbl_path = os.path.join(OUT_LBL_DIR, new_lbl_name)

    if os.path.exists(lbl_path):
        shutil.copy(lbl_path, new_lbl_path)

C:\Users\humus\AppData\Local\Programs\Python\Python310\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Próbki źle oznaczone

In [3]:
import cv2
import os
import yaml
import numpy as np
from pathlib import Path
from ultralytics import YOLO

# ─── KONFIGURACJA ─────────────────────────────────────────────────────────────

MODEL_PATH   = r"D:\Studia\cybAIR\Ambition\rocks_detection\Detection\runs\detect\depthai_model\yolo_rocks_btr_noise\weights\best.pt"
DATA_YAML    = r"D:\Studia\cybAIR\Ambition\rocks_detection\Detection\data.yaml"
OUTPUT_DIR   = r"D:\Studia\cybAIR\Ambition\rocks_detection\Detection\detection_review"

IOU_THRESHOLD   = 0.1   # minimalne IoU żeby uznać detekcję za poprawną
                        # (niskie bo GT bywa dużo większe niż pred – center-point fallback też działa)
CONF_THRESHOLD  = 0.25  # próg pewności modelu

# Kolory BGR
COLOR_TP = (0, 255, 0)    # zielony  – True Positive  (dobra detekcja)
COLOR_FP = (0, 0, 255)    # czerwony – False Positive (wykrył, ale nie powinien)
COLOR_FN = (255, 165, 0)  # pomarańczowy – False Negative (powinien wykryć, nie wykrył)

FONT       = cv2.FONT_HERSHEY_SIMPLEX
FONT_SCALE = 0.55
THICKNESS  = 2

# ─── POMOCNICZE ───────────────────────────────────────────────────────────────

def load_class_names(data_yaml: str) -> list[str]:
    with open(data_yaml, "r") as f:
        data = yaml.safe_load(f)
    return data["names"]


def xywhn_to_xyxy(box, img_w: int, img_h: int) -> tuple[int, int, int, int]:
    """YOLO normalized xywh → pikselowe xyxy."""
    cx, cy, w, h = box
    x1 = int((cx - w / 2) * img_w)
    y1 = int((cy - h / 2) * img_h)
    x2 = int((cx + w / 2) * img_w)
    y2 = int((cy + h / 2) * img_h)
    return x1, y1, x2, y2


def compute_iou(boxA, boxB) -> float:
    ax1, ay1, ax2, ay2 = boxA
    bx1, by1, bx2, by2 = boxB
    ix1 = max(ax1, bx1)
    iy1 = max(ay1, by1)
    ix2 = min(ax2, bx2)
    iy2 = min(ay2, by2)
    inter = max(0, ix2 - ix1) * max(0, iy2 - iy1)
    if inter == 0:
        return 0.0
    areaA = (ax2 - ax1) * (ay2 - ay1)
    areaB = (bx2 - bx1) * (by2 - by1)
    return inter / (areaA + areaB - inter)


def draw_box(img, box, color, label: str):
    x1, y1, x2, y2 = box
    cv2.rectangle(img, (x1, y1), (x2, y2), color, THICKNESS)
    (tw, th), _ = cv2.getTextSize(label, FONT, FONT_SCALE, 1)
    cv2.rectangle(img, (x1, y1 - th - 6), (x1 + tw + 4, y1), color, -1)
    cv2.putText(img, label, (x1 + 2, y1 - 4), FONT, FONT_SCALE, (255, 255, 255), 1, cv2.LINE_AA)


def add_legend(img):
    items = [
        (COLOR_TP, "TP - poprawna detekcja"),
        (COLOR_FP, "FP - falszywy alarm"),
        (COLOR_FN, "FN - pominieta detekcja"),
    ]
    y = 20
    for color, text in items:
        cv2.rectangle(img, (8, y - 12), (28, y + 4), color, -1)
        cv2.putText(img, text, (34, y), FONT, 0.5, (255, 255, 255), 1, cv2.LINE_AA)
        cv2.putText(img, text, (34, y), FONT, 0.5, (0, 0, 0), 2, cv2.LINE_AA)
        cv2.putText(img, text, (34, y), FONT, 0.5, (255, 255, 255), 1, cv2.LINE_AA)
        y += 22


# ─── GŁÓWNA LOGIKA ────────────────────────────────────────────────────────────

def find_label_file(img_path: Path) -> Path | None:
    """Szuka pliku .txt z etykietami obok zdjęcia (YOLO format)."""
    for labels_dir in ["labels", "Labels", "label"]:
        candidate = img_path.parent.parent / labels_dir / img_path.with_suffix(".txt").name
        if candidate.exists():
            return candidate
    # ten sam folder
    candidate = img_path.with_suffix(".txt")
    if candidate.exists():
        return candidate
    return None


def process_dataset(model, data_yaml: str, output_dir: str):
    class_names = load_class_names(data_yaml)

    # Wczytaj ścieżki do zdjęć testowych z data.yaml
    with open(data_yaml, "r") as f:
        data = yaml.safe_load(f)

    yaml_dir = Path(data_yaml).parent

    # Obsługa różnych formatów pól (val / test / valid)
    split_key = next((k for k in ("test", "val", "valid") if k in data), None)
    if split_key is None:
        raise ValueError("data.yaml nie zawiera klucza 'test', 'val' ani 'valid'.")

    images_path = Path(data[split_key])
    if not images_path.is_absolute():
        images_path = yaml_dir / images_path

    image_files = sorted([
        p for p in images_path.rglob("*")
        if p.suffix.lower() in (".jpg", ".jpeg", ".png", ".bmp", ".webp")
    ])

    if not image_files:
        print(f"Brak zdjęć w: {images_path}")
        return

    print(f"Znaleziono {len(image_files)} zdjęć w: {images_path}")

    # Foldery wyjściowe
    base = Path(output_dir)
    dir_correct   = base / "1_correct_detections"
    dir_incorrect = base / "2_incorrect_detections"
    dir_all       = base / "3_all_annotated"
    for d in (dir_correct, dir_incorrect, dir_all):
        d.mkdir(parents=True, exist_ok=True)

    stats = {"tp": 0, "fp": 0, "fn": 0, "correct_imgs": 0, "incorrect_imgs": 0}

    for idx, img_path in enumerate(image_files, 1):
        img = cv2.imread(str(img_path))
        if img is None:
            print(f"  [BŁĄD] Nie można wczytać: {img_path}")
            continue

        h, w = img.shape[:2]
        canvas = img.copy()

        # ── Detekcje modelu ──────────────────────────────────────────────────
        results = model.predict(str(img_path), conf=CONF_THRESHOLD, verbose=False)[0]
        pred_boxes = []
        for box in results.boxes:
            cls_id = int(box.cls[0])
            conf   = float(box.conf[0])
            x1, y1, x2, y2 = map(int, box.xyxy[0].tolist())
            pred_boxes.append({"box": (x1, y1, x2, y2), "cls": cls_id, "conf": conf})

        # ── Ground truth z pliku .txt ────────────────────────────────────────
        gt_boxes = []
        label_file = find_label_file(img_path)
        if label_file:
            with open(label_file, "r") as f:
                for line in f:
                    parts = line.strip().split()
                    if len(parts) < 5:
                        continue
                    cls_id = int(parts[0])
                    coords = list(map(float, parts[1:]))

                    if len(coords) == 4:
                        # Format bbox: cx cy w h (normalized)
                        gt_boxes.append({"box": xywhn_to_xyxy(coords, w, h), "cls": cls_id, "matched": False})
                    else:
                        # Format segmentacji (polygon): x1 y1 x2 y2 ... xN yN (normalized)
                        xs = [coords[i]     * w for i in range(0, len(coords), 2)]
                        ys = [coords[i + 1] * h for i in range(0, len(coords), 2)]
                        x1, y1 = int(min(xs)), int(min(ys))
                        x2, y2 = int(max(xs)), int(max(ys))
                        gt_boxes.append({"box": (x1, y1, x2, y2), "cls": cls_id, "matched": False})

        # ── Dopasowanie TP / FP / FN ─────────────────────────────────────────
        tp_pairs = []
        fp_preds = []

        for pred in pred_boxes:
            best_iou, best_gt = 0.0, None
            px1, py1, px2, py2 = pred["box"]
            pcx, pcy = (px1 + px2) / 2, (py1 + py2) / 2  # środek predykcji

            for gt in gt_boxes:
                if gt["matched"]:
                    continue
                iou = compute_iou(pred["box"], gt["box"])

                # Fallback: środek predykcji leży wewnątrz GT (gdy GT >> pred)
                gx1, gy1, gx2, gy2 = gt["box"]
                center_inside = (gx1 <= pcx <= gx2) and (gy1 <= pcy <= gy2)
                effective_iou = iou if iou > best_iou else (IOU_THRESHOLD if center_inside and iou == 0.0 else iou)

                if effective_iou > best_iou:
                    best_iou, best_gt = effective_iou, gt

            if best_iou >= IOU_THRESHOLD and best_gt is not None:
                best_gt["matched"] = True
                tp_pairs.append((pred, best_gt))
            else:
                fp_preds.append(pred)

        fn_gts = [gt for gt in gt_boxes if not gt["matched"]]

        # ── Rysowanie ────────────────────────────────────────────────────────
        for pred, gt in tp_pairs:
            label = f"TP {class_names[pred['cls']]} {pred['conf']:.2f}"
            draw_box(canvas, pred["box"], COLOR_TP, label)

        for pred in fp_preds:
            label = f"FP {class_names[pred['cls']]} {pred['conf']:.2f}"
            draw_box(canvas, pred["box"], COLOR_FP, label)

        for gt in fn_gts:
            label = f"FN {class_names[gt['cls']]}"
            draw_box(canvas, gt["box"], COLOR_FN, label)

        add_legend(canvas)

        # Statystyki w rogu
        info = f"TP:{len(tp_pairs)}  FP:{len(fp_preds)}  FN:{len(fn_gts)}"
        cv2.putText(canvas, info, (w - 220, h - 10), FONT, 0.6, (0, 0, 0), 3)
        cv2.putText(canvas, info, (w - 220, h - 10), FONT, 0.6, (255, 255, 255), 1)

        stats["tp"] += len(tp_pairs)
        stats["fp"] += len(fp_preds)
        stats["fn"] += len(fn_gts)

        # ── Zapis ────────────────────────────────────────────────────────────
        out_name = f"{img_path.stem}_annot{img_path.suffix}"
        cv2.imwrite(str(dir_all / out_name), canvas)

        has_errors = fp_preds or fn_gts
        if has_errors:
            cv2.imwrite(str(dir_incorrect / out_name), canvas)
            stats["incorrect_imgs"] += 1
        else:
            cv2.imwrite(str(dir_correct / out_name), canvas)
            stats["correct_imgs"] += 1

        print(f"  [{idx:>4}/{len(image_files)}] {img_path.name:40s}  "
              f"TP={len(tp_pairs):2d}  FP={len(fp_preds):2d}  FN={len(fn_gts):2d}  "
              f"→ {'OK' if not has_errors else 'BŁĘDY'}")

    # ── Podsumowanie ──────────────────────────────────────────────────────────
    print("\n" + "═" * 55)
    print("  PODSUMOWANIE")
    print("═" * 55)
    print(f"  Zdjęcia poprawne   : {stats['correct_imgs']}")
    print(f"  Zdjęcia z błędami  : {stats['incorrect_imgs']}")
    print(f"  True Positives     : {stats['tp']}")
    print(f"  False Positives    : {stats['fp']}")
    print(f"  False Negatives    : {stats['fn']}")
    total = stats["tp"] + stats["fp"] + stats["fn"]
    if total:
        precision = stats["tp"] / (stats["tp"] + stats["fp"] + 1e-9)
        recall    = stats["tp"] / (stats["tp"] + stats["fn"] + 1e-9)
        f1        = 2 * precision * recall / (precision + recall + 1e-9)
        print(f"  Precision          : {precision:.3f}")
        print(f"  Recall             : {recall:.3f}")
        print(f"  F1                 : {f1:.3f}")
    print("═" * 55)
    print(f"\n  Wyniki zapisano w: {output_dir}")
    print(f"    ├── 1_correct_detections/   ({stats['correct_imgs']} zdjęć)")
    print(f"    ├── 2_incorrect_detections/ ({stats['incorrect_imgs']} zdjęć)")
    print(f"    └── 3_all_annotated/        ({stats['correct_imgs'] + stats['incorrect_imgs']} zdjęć)")


# ─── ENTRY POINT ──────────────────────────────────────────────────────────────

if __name__ == "__main__":
    print("Ładowanie modelu...")
    model = YOLO(MODEL_PATH)
    process_dataset(model, DATA_YAML, OUTPUT_DIR)

Ładowanie modelu...
Znaleziono 55 zdjęć w: D:\Studia\cybAIR\Ambition\rocks_detection\Detection\test\images
  [   1/55] rgb_00_00320__center_png.rf.39efpQLHS08xuu3wF7HO.png  TP= 1  FP= 1  FN= 0  → BŁĘDY
  [   2/55] rgb_00_00330__center_png.rf.Fmfv99UzytMVjsO1mwHy.png  TP=26  FP=15  FN= 6  → BŁĘDY
  [   3/55] rgb_00_00340__left_png.rf.5odgQw89gPLnH3JPVbUo.png  TP=33  FP=18  FN=14  → BŁĘDY
  [   4/55] rgb_00_00370__right_png.rf.4do60jYQxDfVHgiQ1J4J.png  TP=24  FP= 3  FN=12  → BŁĘDY
  [   5/55] rgb_00_00830__center_png.rf.Zb0zPHNUnc1AgzjnSwv4.png  TP=29  FP= 0  FN= 5  → BŁĘDY
  [   6/55] rgb_00_00860__right_png.rf.9LYgtmZhHjsUj4N0whVo.png  TP= 6  FP= 0  FN= 0  → OK
  [   7/55] rgb_01_00090__right_png.rf.xKqj9CGcjYe47Xti1evF.png  TP=11  FP= 5  FN= 0  → BŁĘDY
  [   8/55] rgb_01_00490__center_png.rf.gaVliaRisMvV60NVxrZZ.png  TP= 6  FP= 0  FN= 4  → BŁĘDY
  [   9/55] rgb_01_00880__center_png.rf.HJzHXQ7OhfwMdZP8XnwH.png  TP= 2  FP= 5  FN= 1  → BŁĘDY
  [  10/55] rgb_02_00860__left_png.rf.6mWStwUs